In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.3/48.9 MB 9.1 MB/s eta 0:00:06
   -- ------------------------------------- 3.4/48.9 MB 8.2 MB/s eta 0:00:06
   ---- ----------------------------------- 5.2/48.9 MB 8.7 MB/s eta 0:00:06
   ------ --------------------------------- 7.9/48.9 MB 9.7 MB/s eta 0:00:05
   -------- ------------------------------- 10.0/48.9 MB 10.0 MB/s eta 0:00:04
   ---------- ----------------------------- 12.3/48.9 MB 10.1 MB/s eta 0:00:04
   ----------- ---------------------------- 14.4/48.9 MB 10.1 MB/s eta 0:00:04
   ------------- -------------------------- 17.0/48.9 MB 10.4 MB/s eta 0:00:04
   --------------- ------------------------ 19.1/48.9 MB 10.4 MB/s eta 0:00:03
   ----------------- ---------------------- 21.5/48.9 MB 10.4 MB/s eta 0:00:03
   ------------------- -------------------- 23.9/48.9 MB 10.5 MB/s eta 0:00:03
   --------------------- ------------------ 26.5/48.9 MB 10.6 MB/s et


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score



In [5]:
# 1. Carregar a base limpa e aplicar a transformação binária
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]



In [6]:
# 2. Isolar X, y e Pesos nas Features Completas
y_bin = df[TARGET_BIN]
weights = df[PESO_COLUNA]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights[mascara_completos].copy()



In [7]:
# 3. Split Estratificado
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)



In [8]:
# 4. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)



In [9]:
# 5. Configuração do Peso de Desbalanceamento para o XGBoost
scale_pos = (y_tr == 0).sum() / (y_tr == 1).sum()



In [10]:
# 6. Espaço de Busca de Hiperparâmetros do XGBoost
param_dist_xgb = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'scale_pos_weight': [scale_pos, scale_pos * 0.8, scale_pos * 1.2]
}



In [11]:
# 7. Execução do RandomizedSearchCV
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Iniciando otimização de hiperparâmetros do XGBoost...")
inicio = time.time()
random_search_xgb.fit(X_tr_proc, y_tr, sample_weight=w_tr)
tempo_total = time.time() - inicio

xgb_otimizado = random_search_xgb.best_estimator_

print(f"\n✓ Otimização concluída em {tempo_total:.2f}s")
print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (XGBOOST) ---")
for param, val in random_search_xgb.best_params_.items():
    print(f"{param}: {val}")



Iniciando otimização de hiperparâmetros do XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✓ Otimização concluída em 96.46s

--- MELHORES HIPERPARÂMETROS ENCONTRADOS (XGBOOST) ---
subsample: 0.9
scale_pos_weight: 2.977017637626938
n_estimators: 300
max_depth: 6
learning_rate: 0.05
gamma: 0
colsample_bytree: 0.8


In [12]:
# 8. Avaliação no Conjunto de Teste
y_pred_xgb = xgb_otimizado.predict(X_te_proc)
y_proba_xgb = xgb_otimizado.predict_proba(X_te_proc)[:, 1]

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO XGBOOST (PONDERADO) ---")
print(classification_report(y_te, y_pred_xgb, sample_weight=w_te, zero_division=0))

# Detalhamento dos Quadrantes da Matriz de Confusão
cm_amostra = confusion_matrix(y_te, y_pred_xgb)
tn, fp, fn, tp = cm_amostra.ravel()
total_amostra = len(y_te)

cm_pop = confusion_matrix(y_te, y_pred_xgb, sample_weight=w_te)
tn_w, fp_w, fn_w, tp_w = cm_pop.ravel()
total_pop = w_te.sum()

print("--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---")
print(f"Verdadeiros Negativos (TN): {tn:5d} | {tn/total_amostra*100:6.2f}% (Sem Depressão - Correto)")
print(f"Falsos Positivos      (FP): {fp:5d} | {fp/total_amostra*100:6.2f}% (Alarme Falso)")
print(f"Falsos Negativos      (FN): {fn:5d} | {fn/total_amostra*100:6.2f}% (Subdiagnóstico)")
print(f"Verdadeiros Positivos (TP): {tp:5d} | {tp/total_amostra*100:6.2f}% (Com Depressão - Capturado)")

print("\n--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---")
print(f"Verdadeiros Negativos (TN): {tn_w:11,.0f} | {tn_w/total_pop*100:6.2f}%")
print(f"Falsos Positivos      (FP): {fp_w:11,.0f} | {fp_w/total_pop*100:6.2f}%")
print(f"Falsos Negativos      (FN): {fn_w:11,.0f} | {fn_w/total_pop*100:6.2f}%")
print(f"Verdadeiros Positivos (TP): {tp_w:11,.0f} | {tp_w/total_pop*100:6.2f}%")

print(f"\nROC-AUC Score (XGBoost): {roc_auc_score(y_te, y_proba_xgb, sample_weight=w_te):.4f}")


--- RELATÓRIO DE CLASSIFICAÇÃO XGBOOST (PONDERADO) ---
              precision    recall  f1-score   support

           0       0.93      0.89      0.91 37773204.546999976
           1       0.66      0.76      0.71 10425277.420000015

    accuracy                           0.86 48198481.96699999
   macro avg       0.79      0.83      0.81 48198481.96699999
weighted avg       0.87      0.86      0.87 48198481.96699999

--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---
Verdadeiros Negativos (TN):  3104 |  70.27% (Sem Depressão - Correto)
Falsos Positivos      (FP):   378 |   8.56% (Alarme Falso)
Falsos Negativos      (FN):   215 |   4.87% (Subdiagnóstico)
Verdadeiros Positivos (TP):   720 |  16.30% (Com Depressão - Capturado)

--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---
Verdadeiros Negativos (TN):  33,660,066 |  69.84%
Falsos Positivos      (FP):   4,113,138 |   8.53%
Falsos Negativos      (FN):   2,493,619 |   5.17%
Verdadeiros Positivos (TP)

### Análise Crítica dos Resultados: XGBoost (Gradient Boosting)

* **Sensibilidade Operacional Elevada (Recall = 76%):** O XGBoost demonstrou superioridade na triagem de indivíduos com depressão (Classe 1), atingindo **76% de Recall** ponderado (contra 71,8% do Random Forest).
* **Minimização do Subdiagnóstico Populacional:** Na projeção populacional expandida, a taxa de falsos negativos (FN) caiu para **5,17%** (2.493.619 pessoas). O algoritmo foi capaz de identificar corretamente **7.931.658 indivíduos afetados** na sociedade.
* **Ganho no F1-Score (0.71):** A combinação de um *Recall* de 76% com uma *Precision* de 66% elevou o F1-score da classe positiva para **0,71**, consolidando o XGBoost como o modelo mais eficiente do ponto de vista de rastreio em saúde pública.
* **Conclusão Metodológica:** Enquanto o Random Forest oferece um ROC-AUC global ligeiramente superior (0.9135 vs. 0.9092), o XGBoost é o modelo recomendado para cenários onde a **prioridade clínica é minimizar casos não-detectados de depressão**.